# RAG Retrieval Evaluation — Comparing Retrieval Pipelines for AstroLearn

Self-contained reproducible evaluation for comparing RAG retrieval strategies on astronomy papers. Each *pipeline* composes a chunker + retriever + optional query rewriter + optional reranker. Pipelines own their own Qdrant collection (collection prefix `rag_retrieval_eval_`) so ablations don't interfere.

This notebook is self-contained. It bundles all source code (originally split across `core/`, `corpus/`, `dataset/`, `retrieval/`, `pipelines/`, `stores/`, `eval/` modules) plus the cached eval results in `rag_retrieval_eval_results/`.

## Pipeline ladder tested

```
Step  Pipeline                              hit@1  hit@5  hit@10  MRR
0     baseline (char-window 1000/200)       0.167  0.500  0.633   0.324
1     tuned recursive (500/250)             0.367  0.567  0.800   0.450
2     + cross-encoder reranker              0.600  0.800  0.900   0.687
```

Also rejected for this dataset: MultiQueryRewriter, HyDE, ContextualChunker — see the comparison table in `rag_retrieval_eval_results/comparison.md` for the negative-result writeup.

## Quick start

1. Install the requirements cell.
2. Set env vars (LiteLLM proxy + Qdrant).
3. Either: (a) jump to **§Final results** for the cached run, or (b) execute every cell.


## §1 Setup

In [ ]:
%pip install --quiet \
    litellm>=1.52.0 qdrant-client>=1.12.0 pypdf>=5.1.0 \
    pyyaml>=6.0 python-dotenv>=1.0.0 pydantic>=2.9.0 \
    httpx>=0.28.0 structlog>=24.4.0 \
    sentence-transformers>=3.0.0  # optional: local embedder + reranker

In [ ]:
import os

# Shared LiteLLM gateway (same proxy as AstroLearn backend).
os.environ.setdefault("LITELLM_BASE_URL", "http://localhost:4000")
os.environ.setdefault("LITELLM_API_KEY", "sk-astrolearn-dev")
os.environ.setdefault("LLM_MODEL", "openai/astrolearn-llm")
os.environ.setdefault("EMBEDDING_MODEL", "openai/astrolearn-embedding")
os.environ.setdefault("LLM_TEMPERATURE", "0.2")
os.environ.setdefault("LLM_TIMEOUT", "60")

# Qdrant — same instance as backend, collections prefixed `rag_retrieval_eval_`.
os.environ.setdefault("QDRANT_URL", "http://localhost:6333")

os.environ.setdefault("LLM_CONCURRENCY", "2")

# Switch to local HuggingFace models for cost-free iteration:
# os.environ['EMBEDDING_BACKEND'] = 'local'
# os.environ['RERANKER_BACKEND'] = 'local'

LITELLM_YAML = '''model_list:
  - model_name: astrolearn-llm
    litellm_params:
      model: groq/llama-3.3-70b-versatile
      api_key: os.environ/GROQ_API_KEY

  - model_name: astrolearn-embedding
    litellm_params:
      model: openai/jina-embeddings-v3
      api_base: https://api.jina.ai/v1
      api_key: os.environ/JINA_API_KEY
      extra_body:
        dimensions: 384

  - model_name: astrolearn-reranker
    litellm_params:
      model: jina_ai/jina-reranker-v2-base-multilingual
      api_key: os.environ/JINA_API_KEY

general_settings:
  master_key: os.environ/LITELLM_MASTER_KEY

litellm_settings:
  drop_params: true
'''
print('OK — env configured. litellm.yaml stored in LITELLM_YAML.')

## §2 LLM client (`core/llm_client.py`)

Async LiteLLM wrapper exposing `complete()` / `embed()` / `rerank()`. Same shape as `backend/core/llm/llm_client.py`.

In [ ]:
"""Tiny LiteLLM wrapper — standalone so the evaluation harness doesn't import backend.

Mirrors the shape of backend/core/llm/llm_client.py but trimmed: no
DB-backed usage tracking, no fallback model orchestration. Just complete()
and embed() through the configured LiteLLM proxy.
"""

from __future__ import annotations

import asyncio
import os
from functools import lru_cache
from typing import Any

import httpx
from litellm import acompletion, aembedding


class LLMClient:
    """Stateless wrapper around `litellm.acompletion` / `aembedding`."""

    def __init__(
        self,
        *,
        model: str | None = None,
        embedding_model: str | None = None,
        reranker_model: str | None = None,
        base_url: str | None = None,
        api_key: str | None = None,
        temperature: float | None = None,
        timeout: float | None = None,
    ) -> None:
        self.model = model or os.environ["LLM_MODEL"]
        self.embedding_model = embedding_model or os.environ["EMBEDDING_MODEL"]
        self.reranker_model = reranker_model or os.environ.get(
            "RERANKER_MODEL", "astrolearn-reranker"
        )
        self.base_url = base_url or os.environ.get("LITELLM_BASE_URL")
        self.api_key = api_key or os.environ.get("LITELLM_API_KEY")
        self.temperature = (
            temperature if temperature is not None
            else float(os.environ.get("LLM_TEMPERATURE", "0.2"))
        )
        self.timeout = (
            timeout if timeout is not None
            else float(os.environ.get("LLM_TIMEOUT", "60"))
        )

    async def complete(
        self,
        messages: list[dict[str, str]],
        *,
        temperature: float | None = None,
        json_mode: bool = False,
        max_tokens: int | None = None,
        **extra: Any,
    ) -> str:
        """Single non-streaming completion → text content."""
        kwargs: dict[str, Any] = {
            "model": self.model,
            "messages": messages,
            "temperature": temperature if temperature is not None else self.temperature,
            "timeout": self.timeout,
        }
        if self.base_url:
            kwargs["api_base"] = self.base_url
        if self.api_key:
            kwargs["api_key"] = self.api_key
        if max_tokens is not None:
            kwargs["max_tokens"] = max_tokens
        if json_mode:
            # drop_params=true in litellm proxy means models that don't
            # support response_format silently ignore it.
            kwargs["response_format"] = {"type": "json_object"}
        kwargs.update(extra)
        response = await acompletion(**kwargs)
        return response.choices[0].message.content or ""

    async def embed(
        self, texts: list[str], *, batch_size: int = 96
    ) -> list[list[float]]:
        """Batch embedding. Routes to local HF model when EMBEDDING_BACKEND=local,
        else through the LiteLLM proxy. Splits into batches of `batch_size`
        to avoid Jina per-call rate/payload caps. Retries on 429/5xx."""
        if not texts:
            return []
        if os.environ.get("EMBEDDING_BACKEND") == "local":
            return await _local_embed(texts)
        out: list[list[float]] = []
        for start in range(0, len(texts), batch_size):
            batch = texts[start:start + batch_size]
            out.extend(await self._embed_one(batch))
        return out

    async def _embed_one(self, batch: list[str]) -> list[list[float]]:
        kwargs: dict[str, Any] = {
            "model": self.embedding_model,
            "input": batch,
            "timeout": self.timeout,
        }
        if self.base_url:
            kwargs["api_base"] = self.base_url
        if self.api_key:
            kwargs["api_key"] = self.api_key

        # Retry transient failures (429, 5xx) with exp backoff.
        last_exc: Exception | None = None
        for attempt in range(5):
            try:
                response = await aembedding(**kwargs)
                return [item["embedding"] for item in response.data]
            except Exception as exc:  # noqa: BLE001
                last_exc = exc
                await asyncio.sleep(2 ** attempt)
        raise RuntimeError(f"embed failed after retries: {last_exc}")

    async def rerank(
        self,
        query: str,
        documents: list[str],
        *,
        top_n: int | None = None,
    ) -> list[tuple[int, float]]:
        """Cross-encoder rerank. Routes to local HF model when
        RERANKER_BACKEND=local, else via LiteLLM proxy /v1/rerank.

        Returns [(orig_index, relevance_score)] sorted desc.
        """
        if not documents:
            return []
        effective_top_n = top_n if top_n is not None else len(documents)
        if os.environ.get("RERANKER_BACKEND") == "local":
            return await _local_rerank(query, documents, top_n=effective_top_n)
        if not self.base_url:
            raise RuntimeError("rerank requires LITELLM_BASE_URL")

        payload = {
            "model": self.reranker_model,
            "query": query,
            "documents": documents,
            "top_n": effective_top_n,
        }
        headers = {"Content-Type": "application/json"}
        if self.api_key:
            headers["Authorization"] = f"Bearer {self.api_key}"

        # Jina free tier rate-limits aggressively (HTTP 429).
        # Exponential backoff: 1s, 2s, 4s, 8s, give up.
        async with httpx.AsyncClient(timeout=self.timeout) as client:
            for attempt in range(5):
                r = await client.post(
                    f"{self.base_url.rstrip('/')}/v1/rerank",
                    json=payload,
                    headers=headers,
                )
                if r.status_code != 429:
                    break
                wait = 2 ** attempt
                print(f"    [rerank-429] backing off {wait}s (attempt {attempt + 1}/5)")
                await asyncio.sleep(wait)
            r.raise_for_status()
            data = r.json()

        # Cohere-compatible: {"results": [{"index": 0, "relevance_score": 0.9}, ...]}
        return [
            (int(item["index"]), float(item["relevance_score"]))
            for item in data.get("results", [])
        ]


@lru_cache(maxsize=1)
def get_llm_client() -> LLMClient:
    """Module-level singleton for CLI scripts."""
    return LLMClient()

## §3 Optional local backend (`core/local_models.py`)

Lets the client route through HuggingFace sentence-transformers instead of the LiteLLM/Jina pipeline. Enabled via `EMBEDDING_BACKEND=local` / `RERANKER_BACKEND=local`.

In [ ]:
"""Local HuggingFace embedder + cross-encoder for cost-free eval iteration.

LLMClient routes here when:
    EMBEDDING_BACKEND=local   (env)
    RERANKER_BACKEND=local    (env)

Trade-off vs LiteLLM/Jina:
  - Free, no rate limit, deterministic across runs
  - First call downloads model (~130MB embedder, ~280MB reranker)
  - CPU embed ~10 ms/chunk, rerank ~50 ms/pair
  - DIFFERENT embedding space from Jina => absolute eval numbers won't
    match backend production; relative pipeline ranking still holds.

Default models chosen to match Jina v3's 384-dim so existing Qdrant
collection schemas stay valid (even though embeddings themselves differ
and require re-indexing on backend switch).
"""

from __future__ import annotations

import asyncio
import os
from functools import lru_cache
from typing import Any

import structlog

_logger = structlog.get_logger(__name__)


@lru_cache(maxsize=1)
def _get_embedder() -> Any:
    """Load + cache the local embedder. Lazy import so torch isn't loaded
    unless we actually use local backend."""
    from sentence_transformers import SentenceTransformer

    model_name = os.environ.get("LOCAL_EMBEDDING_MODEL", "BAAI/bge-small-en-v1.5")
    _logger.info("local_embedder.loading", model=model_name)
    model = SentenceTransformer(model_name)
    _logger.info("local_embedder.loaded", model=model_name, dim=model.get_sentence_embedding_dimension())
    return model


@lru_cache(maxsize=1)
def _get_reranker() -> Any:
    from sentence_transformers import CrossEncoder

    model_name = os.environ.get("LOCAL_RERANKER_MODEL", "BAAI/bge-reranker-base")
    _logger.info("local_reranker.loading", model=model_name)
    model = CrossEncoder(model_name)
    _logger.info("local_reranker.loaded", model=model_name)
    return model


async def embed(texts: list[str]) -> list[list[float]]:
    """Async wrapper over sync sentence-transformers encode."""
    if not texts:
        return []
    model = _get_embedder()

    def _encode() -> list[list[float]]:
        # normalize_embeddings=True: BGE models train with cosine sim, so we
        # normalize once at index/query time. Same convention as Qdrant cosine.
        arr = model.encode(
            texts,
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=False,
        )
        return arr.tolist()

    return await asyncio.to_thread(_encode)


async def rerank(
    query: str, documents: list[str], *, top_n: int
) -> list[tuple[int, float]]:
    """Cross-encoder rerank. Returns [(orig_index, score)] sorted desc."""
    if not documents:
        return []
    model = _get_reranker()

    def _predict() -> list[tuple[int, float]]:
        pairs = [(query, d) for d in documents]
        scores = model.predict(pairs, show_progress_bar=False)
        ranked = sorted(enumerate(scores), key=lambda x: -float(x[1]))
        return [(i, float(s)) for i, s in ranked[:top_n]]

    return await asyncio.to_thread(_predict)

## §4 Corpus — download + parse

6 arXiv astronomy papers downloaded to `corpus/papers/`, then parsed into `(doc_id, page, text)` tuples with deterministic UUIDv5 ids.

In [ ]:
"""Download the arxiv astronomy corpus to corpus/papers/."""

from __future__ import annotations

import urllib.error
import urllib.request
from pathlib import Path

# arxiv id -> filename slug. Mix of papers on LLMs-for-astro, RAG-for-astro,
# JWST early galaxies, black hole formation, and one survey paper. Keep this
# stable so eval datasets stay comparable across versions.
PAPERS: dict[str, str] = {
    "llm_astronomy_eval":   "https://arxiv.org/pdf/2405.20389",
    "rag_astronomy_agents": "https://arxiv.org/pdf/2507.07155",
    "astromlabai":          "https://arxiv.org/pdf/2411.09012",
    "early_galaxies_jwst":  "https://arxiv.org/pdf/2406.15548",
    "black_hole_formation": "https://arxiv.org/pdf/2406.13072",
    "astro_wrapped_2025":   "https://arxiv.org/pdf/2602.12303",
}


def repo_root() -> Path:
    # corpus/download.py -> repo root is parent.parent
    return Path(__file__).resolve().parents[1]


def papers_dir() -> Path:
    out = repo_root() / "corpus" / "papers"
    out.mkdir(parents=True, exist_ok=True)
    return out


def download_all() -> tuple[int, list[str]]:
    """Download missing PDFs. Returns (success_count, failed_names)."""
    out_dir = papers_dir()
    print(f"[*] Downloading to {out_dir}")
    failed: list[str] = []
    ok = 0
    for name, url in PAPERS.items():
        target = out_dir / f"{name}.pdf"
        if target.exists() and target.stat().st_size > 1024:
            print(f"  [skip] {name}.pdf ({target.stat().st_size // 1024} KB)")
            ok += 1
            continue
        print(f"  [..] {name}: {url}")
        try:
            # arxiv refuses default urllib UA.
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=60) as resp:
                target.write_bytes(resp.read())
            print(f"  [ok] {name}.pdf ({target.stat().st_size // 1024} KB)")
            ok += 1
        except (urllib.error.URLError, TimeoutError) as exc:
            print(f"  [fail] {name}: {exc}")
            failed.append(name)
    return ok, failed

In [ ]:
"""Parse PDFs into a deterministic in-memory corpus + manifest.

Corpus parsing is pipeline-independent: every pipeline gets the same
(doc_id, page_number, text) tuples. doc_ids are deterministic (UUIDv5 from
filename) so re-parsing produces stable ids — eval datasets stay valid
across reruns.
"""

from __future__ import annotations

import json
import uuid
from dataclasses import asdict, dataclass, field
from pathlib import Path


# Stable namespace: same filename -> same doc_id forever.
_DOC_ID_NAMESPACE = uuid.UUID("c1a2b3d4-e5f6-7890-abcd-ef1234567890")


@dataclass
class CorpusPage:
    page_number: int        # 1-indexed
    text: str
    char_offset: int        # start position in CorpusDoc.full_text


@dataclass
class CorpusDoc:
    doc_id: uuid.UUID
    filename: str
    page_count: int
    full_text: str
    pages: list[CorpusPage] = field(default_factory=list)


def doc_id_for(filename: str) -> uuid.UUID:
    """Deterministic doc id from filename — survives re-parses."""
    return uuid.uuid5(_DOC_ID_NAMESPACE, filename)


def manifest_path() -> Path:
    return repo_root() / "corpus" / "manifest.json"


def parse_one(pdf: Path) -> CorpusDoc:
    """Parse a single PDF; pages with empty text are kept (with their offset)."""
    from pypdf import PdfReader

    reader = PdfReader(str(pdf))
    pages: list[CorpusPage] = []
    full_parts: list[str] = []
    offset = 0
    for i, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        pages.append(CorpusPage(page_number=i, text=text, char_offset=offset))
        full_parts.append(text)
        offset += len(text) + 1  # +1 for "\n" joiner
    return CorpusDoc(
        doc_id=doc_id_for(pdf.name),
        filename=pdf.name,
        page_count=len(pages),
        full_text="\n".join(full_parts),
        pages=pages,
    )


def parse_corpus() -> list[CorpusDoc]:
    """Parse every PDF in corpus/papers/ and write a manifest."""
    pdfs = sorted(papers_dir().glob("*.pdf"))
    if not pdfs:
        raise FileNotFoundError(
            f"No PDFs in {papers_dir()}. Run `run download` first."
        )

    docs: list[CorpusDoc] = []
    for pdf in pdfs:
        print(f"  [parse] {pdf.name}")
        doc = parse_one(pdf)
        docs.append(doc)
        print(f"    -> {doc.page_count} pages, {len(doc.full_text)} chars")

    # Manifest excludes raw text (too large); just metadata + doc_id mapping.
    manifest = {
        "documents": [
            {
                "doc_id": str(d.doc_id),
                "filename": d.filename,
                "page_count": d.page_count,
                "total_chars": len(d.full_text),
            }
            for d in docs
        ]
    }
    manifest_path().write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    print(f"[+] Wrote {manifest_path()} ({len(docs)} docs)")
    return docs


def load_manifest() -> list[dict]:
    """Return [{doc_id, filename, ...}, ...] from corpus/manifest.json."""
    p = manifest_path()
    if not p.exists():
        raise FileNotFoundError(
            f"{p} missing. Run `run parse-corpus` first."
        )
    return json.loads(p.read_text(encoding="utf-8"))["documents"]


def page_dict(doc: CorpusDoc) -> dict[int, str]:
    """Convenience: {page_number: text} for fast lookup."""
    return {p.page_number: p.text for p in doc.pages}


def doc_to_jsonable(doc: CorpusDoc) -> dict:
    """For debugging / interactive notebooks."""
    return {
        "doc_id": str(doc.doc_id),
        "filename": doc.filename,
        "page_count": doc.page_count,
        "pages": [asdict(p) for p in doc.pages],
    }

## §5 Dataset schemas + generator

`GoldenDataset` = list of `GoldenQuery` (each pointing at a `GoldenChunk` constraint: `(doc_id, page ± tolerance, contains_substring)`). The generator prompts an LLM per page for one grounded Q&A.

In [ ]:
"""Golden dataset + eval result schemas."""

from __future__ import annotations

import uuid
from datetime import datetime
from typing import Any

from pydantic import BaseModel, ConfigDict, Field


class GoldenChunk(BaseModel):
    """A region a query should retrieve. Fuzzy on purpose so datasets
    survive chunker changes."""

    doc_id: uuid.UUID
    page: int | None = Field(None, ge=1)
    page_tolerance: int = Field(1, ge=0, le=5)
    contains: str | None = Field(None, max_length=500)


class GoldenQuery(BaseModel):
    id: str = Field(..., min_length=1, max_length=64)
    question: str = Field(..., min_length=1, max_length=2000)
    relevant: list[GoldenChunk] = Field(..., min_length=1)
    tags: list[str] = Field(default_factory=list)


class GoldenDataset(BaseModel):
    name: str
    description: str = ""
    queries: list[GoldenQuery] = Field(..., min_length=1)


class QueryResult(BaseModel):
    query_id: str
    question: str
    n_relevant: int
    n_retrieved: int
    hit_ranks: list[int]                # 1-indexed ranks of goldens found
    first_hit_rank: int | None


class EvalResult(BaseModel):
    """One eval run, serialised to JSON for diffing."""

    model_config = ConfigDict(json_schema_extra={"description": "RAG retrieval eval result"})

    pipeline_name: str
    dataset_name: str
    dataset_path: str
    timestamp: datetime
    top_k: int
    pipeline_config: dict[str, Any]
    queries: list[QueryResult]

    hit_rate_at_1: float
    hit_rate_at_5: float
    hit_rate_at_10: float
    recall_at_5: float
    recall_at_10: float
    mrr: float

    @property
    def n_queries(self) -> int:
        return len(self.queries)


def metric_keys() -> list[str]:
    return [
        "hit_rate_at_1",
        "hit_rate_at_5",
        "hit_rate_at_10",
        "recall_at_5",
        "recall_at_10",
        "mrr",
    ]

In [ ]:
"""Generate a golden dataset by prompting the LLM per page.

For each parsed doc, sample N content-rich pages, ask the LLM for ONE
grounded question + a distinctive key_phrase that must appear verbatim
on the page (whitespace-normalised). Skip pages that don't yield a
verifiable Q&A.

Uses json_mode through the LiteLLM proxy. Concurrency capped (LLM_CONCURRENCY)
so Groq's free tier doesn't 429.
"""

from __future__ import annotations

import asyncio
import json
import os
import uuid
from pathlib import Path
from typing import Any

import yaml


_MIN_PAGE_CHARS = 800
_MAX_PAGE_CHARS_IN_PROMPT = 6000
_KEY_PHRASE_MIN = 15           # 15 chars is short but still distinctive ("Learning rate 1e-4")
_KEY_PHRASE_MAX = 120

_PROMPT = """\
You are creating a RAG retrieval-evaluation question from ONE page of an \
astronomy paper.

GOOD question types (write one of these):
- Numeric / quantitative facts: "What is the peak learning rate used in training?"
- Method or technique: "What algorithm does the paper use for source detection?"
- Specific definitions: "How does the paper define the LCI metric?"
- Comparison / claim: "Which model achieves higher accuracy on CosmoPaperQA?"
- Architectural detail: "What attention mechanism does AstroSage use?"

BAD question types - SET skip=true if this page only supports these:
- Reference/citation: "What is the title of the paper by Brown et al. 2020?"
- Generic table/figure captions: "What does Table 3 show?"
- About the paper's existence: "Who are the authors?", "What journal published this?"
- Boilerplate: acknowledgments, license, copyright, author affiliations
- Pure references list, table of contents, index

Rules:
- The question must target a SPECIFIC fact stated in the page text below.
- The question must be answerable from this page; do NOT invent.
- Pick a substring of {key_min}-{key_max} characters from the page that \
contains the answer. COPY IT EXACTLY from the text - no rewording, no \
punctuation changes. Whitespace will be normalised when checking.
- The key_phrase should be distinctive (avoid generic phrases like "the model" \
or "the result"). Include a number, name, or specific term where possible.
- If unsure or page is unsuitable, prefer skip=true over a low-quality question.

Output ONLY valid JSON:
{{"question": "...", "key_phrase": "...", "skip": false}}

Page text:
\"\"\"
{page_text}
\"\"\"
"""


def repo_root() -> Path:
    return Path(__file__).resolve().parents[1]


def datasets_dir() -> Path:
    out = repo_root() / "dataset" / "datasets"
    out.mkdir(parents=True, exist_ok=True)
    return out


def _normalize(text: str) -> str:
    """Whitespace + case normalisation. Survives most pypdf glitches."""
    return " ".join(text.split()).lower()


def _strip_punct(text: str) -> str:
    """Stronger normalisation: drop punctuation entirely. Last-resort fallback."""
    out = []
    for ch in text.lower():
        if ch.isalnum() or ch.isspace():
            out.append(ch)
    return " ".join("".join(out).split())


def _select_pages(
    pages: list[tuple[int, str]], per_doc: int
) -> list[tuple[int, str]]:
    """Pick per_doc pages evenly spaced, trimming 10% off each end."""
    if not pages:
        return []
    n = len(pages)
    if n >= 10:
        margin = max(1, n // 10)
        candidates = pages[margin : n - margin]
    else:
        candidates = pages
    if len(candidates) <= per_doc:
        return candidates
    step = len(candidates) / per_doc
    return [candidates[int(i * step)] for i in range(per_doc)]


async def _ask_one(
    llm: LLMClient, page_text: str, sem: asyncio.Semaphore
) -> dict[str, Any] | None:
    """Return parsed dict or None on failure (LLM error / invalid JSON)."""
    prompt = _PROMPT.format(
        key_min=_KEY_PHRASE_MIN,
        key_max=_KEY_PHRASE_MAX,
        page_text=page_text[:_MAX_PAGE_CHARS_IN_PROMPT],
    )
    async with sem:
        try:
            raw = await llm.complete(
                [{"role": "user", "content": prompt}],
                temperature=0.4,
                json_mode=True,
            )
        except Exception as exc:  # noqa: BLE001
            return {"_error": f"llm: {exc}"}
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"_error": "invalid json"}


def _validate(parsed: dict[str, Any], page_text: str) -> tuple[str, str] | None:
    if parsed.get("skip"):
        return None
    if "_error" in parsed:
        return None
    question = (parsed.get("question") or "").strip()
    key_phrase = (parsed.get("key_phrase") or "").strip()
    if not question or not key_phrase:
        return None
    if not (_KEY_PHRASE_MIN <= len(key_phrase) <= _KEY_PHRASE_MAX * 2):
        return None
    # Try increasingly permissive normalisations.
    norm_page = _normalize(page_text)
    norm_key = _normalize(key_phrase)
    if norm_key in norm_page:
        return question, key_phrase
    if _strip_punct(key_phrase) in _strip_punct(page_text):
        return question, key_phrase
    return None


def _content_pages(doc: CorpusDoc) -> list[tuple[int, str]]:
    return [(p.page_number, p.text) for p in doc.pages if len(p.text) >= _MIN_PAGE_CHARS]


async def generate(
    *,
    per_doc: int,
    name: str,
    out_path: Path | None = None,
) -> Path:
    """Generate the dataset; returns path written."""
    docs = parse_corpus()
    llm = get_llm_client()
    concurrency = int(os.environ.get("LLM_CONCURRENCY", "2"))
    sem = asyncio.Semaphore(concurrency)

    all_queries: list[dict[str, Any]] = []
    for doc in docs:
        pages = _content_pages(doc)
        if not pages:
            print(f"  [{doc.filename}] no content-rich pages, skipping")
            continue
        selected = _select_pages(pages, per_doc)
        print(f"  [{doc.filename}] {len(pages)} content-rich, sampling {len(selected)}")

        # Concurrent within a doc; semaphore caps global parallelism.
        results = await asyncio.gather(
            *(_ask_one(llm, text, sem) for _, text in selected)
        )

        for (page_num, page_text), parsed in zip(selected, results, strict=True):
            if parsed is None:
                print(f"    [skip page {page_num}] llm error")
                continue
            validated = _validate(parsed, page_text)
            if validated is None:
                err = parsed.get("_error") or "unverifiable"
                print(f"    [skip page {page_num}] {err}")
                continue
            question, key_phrase = validated
            all_queries.append(
                {
                    "id": "",  # filled below
                    "question": question,
                    "tags": [Path(doc.filename).stem],
                    "relevant": [
                        {
                            "doc_id": str(doc.doc_id),
                            "page": page_num,
                            "page_tolerance": 1,
                            "contains": key_phrase[:480],
                        }
                    ],
                }
            )
            print(f"    [ok page {page_num}] {question[:70]}")

    if not all_queries:
        raise RuntimeError("No queries generated.")

    for i, q in enumerate(all_queries, start=1):
        q["id"] = f"q{i:02d}"

    dataset = {
        "name": name,
        "description": (
            f"Per-page LLM-generated grounded questions ({len(all_queries)} total). "
            "Review and prune before treating as ground truth."
        ),
        "queries": all_queries,
    }

    out = out_path or (datasets_dir() / f"{name}.yaml")
    with out.open("w", encoding="utf-8") as f:
        yaml.safe_dump(dataset, f, sort_keys=False, allow_unicode=True, width=120)
    print(f"\n[+] Wrote {out} ({len(all_queries)} queries)")
    return out

## §6 Qdrant store (`stores/qdrant_store.py`)

Thin async wrapper, one instance per pipeline. Each pipeline owns its own collection so reindexing one doesn't disturb others.

In [ ]:
"""Per-pipeline Qdrant collection wrapper.

Each pipeline gets its own collection (`rag_retrieval_eval_<name>`) so you can:
  - reindex one pipeline without touching others
  - drop a pipeline's data with one call
  - compare side-by-side at search time
"""

from __future__ import annotations

import os
import uuid
from dataclasses import dataclass
from typing import Any

from qdrant_client import AsyncQdrantClient
from qdrant_client.models import (
    Distance,
    FieldCondition,
    Filter,
    MatchValue,
    PointStruct,
    VectorParams,
)

# Stable namespace: same chunk_id -> same point UUID across reindexes.
_POINT_ID_NAMESPACE = uuid.UUID("a7b3c2d1-e4f5-6789-0abc-def123456789")


@dataclass
class StoredChunk:
    chunk_id: str       # f"{doc_id}:{i}" or pipeline-specific
    doc_id: uuid.UUID
    text: str
    vector: list[float]
    payload: dict[str, Any]  # additional metadata (page, parent_id, etc.)


@dataclass
class RetrievedHit:
    chunk_id: str
    doc_id: uuid.UUID
    text: str
    score: float           # [0,1] (cosine clamped)
    payload: dict[str, Any]


class QdrantStore:
    """Thin async wrapper, one instance per pipeline (binds a collection name)."""

    def __init__(
        self,
        collection: str,
        *,
        url: str | None = None,
        client: AsyncQdrantClient | None = None,
    ) -> None:
        self.collection = collection
        self._client = client or AsyncQdrantClient(
            url=url or os.environ.get("QDRANT_URL", "http://localhost:6333")
        )

    @property
    def client(self) -> AsyncQdrantClient:
        return self._client

    async def ensure_collection(self, *, dimension: int) -> None:
        """Idempotent create with cosine distance."""
        if await self._client.collection_exists(self.collection):
            return
        await self._client.create_collection(
            collection_name=self.collection,
            vectors_config=VectorParams(size=dimension, distance=Distance.COSINE),
        )

    async def drop(self) -> None:
        """Delete the collection if it exists (no-op otherwise)."""
        if await self._client.collection_exists(self.collection):
            await self._client.delete_collection(self.collection)

    async def upsert(self, chunks: list[StoredChunk]) -> int:
        if not chunks:
            return 0
        points = [
            PointStruct(
                id=str(uuid.uuid5(_POINT_ID_NAMESPACE, c.chunk_id)),
                vector=c.vector,
                payload={
                    "chunk_id": c.chunk_id,
                    "doc_id": str(c.doc_id),
                    "text": c.text,
                    **c.payload,
                },
            )
            for c in chunks
        ]
        await self._client.upsert(collection_name=self.collection, points=points)
        return len(points)

    async def query(
        self,
        vector: list[float],
        *,
        top_k: int = 10,
        filter_payload: dict[str, Any] | None = None,
    ) -> list[RetrievedHit]:
        """Vector search; optional exact-match payload filter (AND of keys)."""
        q_filter: Filter | None = None
        if filter_payload:
            q_filter = Filter(
                must=[
                    FieldCondition(key=k, match=MatchValue(value=v))
                    for k, v in filter_payload.items()
                ]
            )
        # qdrant-client >=1.10 renamed search -> query_points; old `search`
        # is removed in newer versions.
        response = await self._client.query_points(
            collection_name=self.collection,
            query=vector,
            limit=top_k,
            query_filter=q_filter,
            with_payload=True,
        )
        return [_to_hit(h) for h in response.points]


def _to_hit(h: Any) -> RetrievedHit:
    payload = h.payload or {}
    return RetrievedHit(
        chunk_id=str(payload.get("chunk_id") or h.id),
        doc_id=uuid.UUID(payload["doc_id"]),
        text=str(payload.get("text") or ""),
        # Cosine in qdrant is [-1, 1]; clamp to [0, 1] for sanity.
        score=max(0.0, min(1.0, float(h.score))),
        payload={
            k: v for k, v in payload.items() if k not in {"chunk_id", "doc_id", "text"}
        },
    )

## §7 Chunkers

Three strategies behind the `BaseChunker` interface:
- `CharWindowChunker` — dumb fixed-size windows (baseline).
- `RecursiveChunker` — hand-written recursive character splitter; respects paragraph / sentence boundaries.
- `ContextualChunker` — wraps another chunker, prepends per-doc context (filename + title) before embedding.

In [ ]:
"""Chunker base interface — pure functions over CorpusDoc."""

from __future__ import annotations

import uuid
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Any



@dataclass
class Chunk:
    """A chunk emitted by a chunker, ready to be embedded + upserted."""

    chunk_id: str                          # stable within a (pipeline, corpus)
    doc_id: uuid.UUID
    text: str
    page: int | None                       # 1-indexed, None if cross-page
    char_offset: int                       # absolute offset in full_text
    metadata: dict[str, Any] = field(default_factory=dict)


class BaseChunker(ABC):
    """Strategy: turn a CorpusDoc into a list of Chunks."""

    name: str = "base"

    @abstractmethod
    def chunk(self, doc: CorpusDoc) -> list[Chunk]:
        """Chunk one document; pure, deterministic, no I/O."""

In [ ]:
"""Char-window chunker — mirrors backend/workers/notebook_worker._build_chunks.

Baseline for comparison: dumb fixed-size windows with overlap. No respect
for sentence / paragraph boundaries. Lower bound on chunking quality.
"""

from __future__ import annotations

from typing import Any



class CharWindowChunker(BaseChunker):
    name = "char_window"

    def __init__(self, *, size: int = 1000, overlap: int = 200) -> None:
        if overlap >= size:
            raise ValueError(f"overlap ({overlap}) must be < size ({size})")
        self.size = size
        self.overlap = overlap

    def chunk(self, doc: CorpusDoc) -> list[Chunk]:
        text = doc.full_text
        if not text.strip():
            return []

        chunks: list[Chunk] = []
        step = self.size - self.overlap
        idx = 0
        start = 0
        n = len(text)
        while start < n:
            end = min(start + self.size, n)
            slice_text = text[start:end]
            if slice_text.strip():
                meta: dict[str, Any] = {"chunker": self.name}
                page = _page_for_offset(start, doc.pages)
                chunks.append(
                    Chunk(
                        chunk_id=f"{doc.doc_id}:{idx}",
                        doc_id=doc.doc_id,
                        text=slice_text,
                        page=page,
                        char_offset=start,
                        metadata=meta,
                    )
                )
                idx += 1
            if end >= n:
                break
            start += step
        return chunks


def _page_for_offset(offset: int, pages: list[CorpusPage]) -> int | None:
    """Largest page whose char_offset <= offset. Linear scan; pages are sorted."""
    if not pages:
        return None
    selected: int | None = None
    for p in pages:
        if p.char_offset <= offset:
            selected = p.page_number
        else:
            break
    return selected

In [ ]:
"""Hand-written recursive character splitter.

Splits text on a hierarchy of separators (`\\n\\n` -> `\\n` -> `. ` -> ` `),
recursing into oversized pieces, then merging small consecutive pieces back
into chunks of ~`size` characters with `overlap`-character overlap.

Compared to char_window:
  - never cuts mid-word
  - preserves paragraph / sentence boundaries when possible
  - same chunk_id scheme (`{doc_id}:{i}`) and char_offset tracking so
    page mapping + matcher logic stays identical
"""

from __future__ import annotations

from typing import Any


# Default separators, ordered from coarsest to finest. The empty string at
# the end forces a hard character split when nothing else fits.
_DEFAULT_SEPARATORS: tuple[str, ...] = ("\n\n", "\n", ". ", " ", "")


class RecursiveChunker(BaseChunker):
    name = "recursive"

    def __init__(
        self,
        *,
        size: int = 1000,
        overlap: int = 150,
        separators: tuple[str, ...] = _DEFAULT_SEPARATORS,
    ) -> None:
        if overlap >= size:
            raise ValueError(f"overlap ({overlap}) must be < size ({size})")
        self.size = size
        self.overlap = overlap
        self.separators = separators

    # --- public ---

    def chunk(self, doc: CorpusDoc) -> list[Chunk]:
        if not doc.full_text.strip():
            return []
        pieces = self._split(doc.full_text, list(self.separators))
        pieces = self._merge(pieces, self._best_separator(doc.full_text))

        chunks: list[Chunk] = []
        cursor = 0
        for i, piece in enumerate(pieces):
            piece = piece.strip()
            if not piece:
                continue
            offset = self._locate(doc.full_text, piece, cursor)
            chunks.append(
                Chunk(
                    chunk_id=f"{doc.doc_id}:{i}",
                    doc_id=doc.doc_id,
                    text=piece,
                    page=_page_for_offset(offset, doc.pages),
                    char_offset=offset,
                    metadata={"chunker": self.name},
                )
            )
            # Advance past most of this chunk so the next find() doesn't
            # re-locate inside it (overlap-safe).
            cursor = max(cursor, offset + max(1, len(piece) - self.overlap))
        return chunks

    # --- internals ---

    def _best_separator(self, text: str) -> str:
        """The first separator from self.separators that occurs in text."""
        for sep in self.separators:
            if sep == "" or sep in text:
                return sep
        return ""

    def _split(self, text: str, separators: list[str]) -> list[str]:
        """Recursive split: returns pieces all <= size where possible."""
        if not text:
            return []
        if len(text) <= self.size:
            return [text]

        # Pick the coarsest separator that's actually in the text.
        separator = ""
        new_separators: list[str] = []
        for i, sep in enumerate(separators):
            if sep == "":
                separator = sep
                break
            if sep in text:
                separator = sep
                new_separators = separators[i + 1:]
                break

        if separator == "":
            return self._hard_split(text)

        splits = text.split(separator)
        out: list[str] = []
        for s in splits:
            if len(s) < self.size:
                out.append(s)
            elif new_separators:
                out.extend(self._split(s, new_separators))
            else:
                out.extend(self._hard_split(s))
        return out

    def _merge(self, splits: list[str], separator: str) -> list[str]:
        """Pack small adjacent splits into ~size chunks with overlap."""
        chunks: list[str] = []
        current: list[str] = []
        current_len = 0
        sep_len = len(separator)

        for s in splits:
            if not s:
                continue
            extra = (sep_len if current else 0) + len(s)
            if current_len + extra <= self.size:
                if current:
                    current.append(separator)
                    current_len += sep_len
                current.append(s)
                current_len += len(s)
                continue

            # Flush current; start a new chunk seeded with overlap tail.
            if current:
                chunks.append("".join(current))
            if self.overlap > 0 and chunks:
                tail = chunks[-1][-self.overlap :]
                # If s + tail still exceeds size, just take s.
                if len(tail) + sep_len + len(s) <= self.size:
                    current = [tail, separator, s] if separator else [tail, s]
                    current_len = len(tail) + sep_len + len(s)
                else:
                    current = [s]
                    current_len = len(s)
            else:
                current = [s]
                current_len = len(s)

        if current:
            chunks.append("".join(current))
        return chunks

    def _hard_split(self, text: str) -> list[str]:
        """Character-level fallback when no separator helps."""
        step = self.size - self.overlap
        out: list[str] = []
        for i in range(0, len(text), step):
            piece = text[i : i + self.size]
            if piece:
                out.append(piece)
            if i + self.size >= len(text):
                break
        return out

    @staticmethod
    def _locate(text: str, piece: str, start: int) -> int:
        """Find piece at or after `start` in text; fall back to start."""
        pos = text.find(piece, start)
        if pos < 0:
            # Could happen if merge added overlap text that doesn't appear
            # contiguously in original. Try without the overlap tail.
            short = piece[len(piece) // 4 :]
            pos = text.find(short, start)
            if pos < 0:
                return start
            return max(start, pos - len(piece) // 4)
        return pos


def _page_for_offset(offset: int, pages: list[CorpusPage]) -> int | None:
    if not pages:
        return None
    selected: int | None = None
    for p in pages:
        if p.char_offset <= offset:
            selected = p.page_number
        else:
            break
    return selected

In [ ]:
"""Contextual chunker — wraps another chunker and prepends per-doc context.

Implements the simplest variant of Anthropic's "contextual retrieval"
(https://www.anthropic.com/news/contextual-retrieval): prepend each chunk
with a short description of the document it came from before embedding.

Why it helps:
  - For a multi-doc corpus, a chunk saying "the model achieves 80.9%" is
    ambiguous across papers. Prepending "From AstroSage-Llama-3.1-8B paper:"
    grounds the embedding in the right paper.
  - Recovers global context the chunker stripped away (paper title, key terms).

Here we use a minimal heuristic: derive a short context string from the
filename (stem) plus the document's first non-trivial line (likely title).
Cheap to compute, no extra LLM calls.
"""

from __future__ import annotations



def _derive_context(doc: CorpusDoc) -> str:
    """Short context string: filename stem + first non-empty line of doc."""
    # Filename stem as the canonical paper id (matches the slug we choose
    # when downloading). Replaces underscores so embeddings tokenize cleanly.
    stem = doc.filename.removesuffix(".pdf").replace("_", " ")

    # First non-trivial line: usually the title.
    title = ""
    for line in doc.full_text.splitlines():
        s = line.strip()
        if len(s) >= 20:  # skip headers, page numbers, short metadata lines
            title = s[:200]
            break

    if title:
        return f"From paper '{stem}' (title: {title})."
    return f"From paper '{stem}'."


class ContextualChunker(BaseChunker):
    """Wraps another chunker and prepends per-doc context to each chunk's text."""

    name = "contextual"

    def __init__(self, *, inner: BaseChunker) -> None:
        self.inner = inner
        self.name = f"contextual_{inner.name}"

    def chunk(self, doc: CorpusDoc) -> list[Chunk]:
        prefix = _derive_context(doc)
        chunks = self.inner.chunk(doc)
        out: list[Chunk] = []
        for c in chunks:
            # Embed-time text includes the prefix; original char_offset still
            # points into doc.full_text so page mapping is unchanged.
            out.append(
                Chunk(
                    chunk_id=c.chunk_id,
                    doc_id=c.doc_id,
                    text=f"{prefix}\n\n{c.text}",
                    page=c.page,
                    char_offset=c.char_offset,
                    metadata={**c.metadata, "chunker": self.name, "context_prefix": prefix},
                )
            )
        return out

## §8 Query rewriters

- `MultiQueryRewriter` — LLM produces N paraphrases of the question.
- `HyDERewriter` — LLM writes a hypothetical answer paragraph; embed THAT alongside the original.

In [ ]:
"""QueryRewriter interface: one question -> N retrieval queries."""

from __future__ import annotations

from abc import ABC, abstractmethod


class BaseQueryRewriter(ABC):
    """Strategy: expand a single user question into multiple retrieval queries."""

    name: str = "base"

    @abstractmethod
    async def rewrite(self, query: str) -> list[str]:
        """Return [original, variant1, variant2, ...]. Original is always first."""

In [ ]:
"""Multi-query rewriter: LLM produces N paraphrases of one question.

Astronomy-aware: instructs the model to expand catalog IDs / common names
(Andromeda <-> M31 <-> NGC 224) and technical short forms. The original
question is always kept at position 0 so we don't lose ground by rewriting.
"""

from __future__ import annotations

import json


_PROMPT = """You're a query rewriter for a RAG system over astronomy papers.

Given the user's question, produce {n} alternative phrasings that:
1. Preserve the exact intent (do NOT change what is being asked).
2. Use DIFFERENT vocabulary - synonyms, technical terms, common names.
3. For astronomy specifically, expand entity names where applicable:
   - "Andromeda" <-> "M31" <-> "NGC 224"
   - "black hole" <-> "BH" <-> "compact object"
   - "JWST" <-> "James Webb Space Telescope"
   - "supernova" <-> "SN"
4. Vary specificity: one slightly more specific, one slightly more general.

Do NOT include the original question - only alternatives.

Output ONLY valid JSON: {{"queries": ["alt1", "alt2", ...]}}

Original question: {question}
"""


class MultiQueryRewriter(BaseQueryRewriter):
    name = "multi_query"

    def __init__(
        self,
        llm: LLMClient,
        *,
        n_variants: int = 3,
        temperature: float = 0.5,
    ) -> None:
        self.llm = llm
        self.n_variants = n_variants
        self.temperature = temperature

    async def rewrite(self, query: str) -> list[str]:
        prompt = _PROMPT.format(n=self.n_variants, question=query)
        try:
            raw = await self.llm.complete(
                [{"role": "user", "content": prompt}],
                temperature=self.temperature,
                json_mode=True,
            )
        except Exception as exc:  # noqa: BLE001 — degrade to single-query
            print(f"    [rewriter-fail] {exc}; using original only")
            return [query]
        try:
            parsed = json.loads(raw)
        except json.JSONDecodeError:
            return [query]
        variants = parsed.get("queries") or []
        cleaned = [v.strip() for v in variants if isinstance(v, str) and v.strip()]
        # De-dupe and drop anything equal to the original (case-insensitive).
        original_low = query.strip().lower()
        out: list[str] = [query]
        seen: set[str] = {original_low}
        for v in cleaned:
            low = v.lower()
            if low in seen:
                continue
            seen.add(low)
            out.append(v)
        return out

In [ ]:
"""HyDE: Hypothetical Document Embeddings.

Ask the LLM to generate a short paragraph that *could* answer the question,
written in the style of the indexed corpus. Then embed THAT paragraph (instead
of, or alongside, the original question) for retrieval.

Why it helps:
  - Questions and documents live in different vocabulary spaces ("what is the
    learning rate?" vs "Adam optimizer with lr=1.5e-4"). A hypothetical
    answer is closer to corpus language than the question is.
  - Even if the LLM's guess is factually wrong, it tends to use the right
    technical terms, units, and sentence shape — that's what embeddings
    latch onto.

Reference: Gao et al. 2022 "Precise Zero-Shot Dense Retrieval without
Relevance Labels" (https://arxiv.org/abs/2212.10496).
"""

from __future__ import annotations


_PROMPT = """\
You are a query expansion tool for a RAG system over astronomy / astrophysics \
research papers.

Given the user's question, write a SHORT hypothetical paragraph (1-3 sentences, \
maximum 60 words) that could plausibly appear in the paper as an answer.

Rules:
- Write in the voice of an astronomy paper: passive voice, technical \
vocabulary, specific units (M_sun, Mpc, eV, ...) where relevant.
- BE SPECIFIC. If you can guess a plausible numeric value, name, telescope, \
or technique, include it. Being wrong is fine — purpose is to match the \
text style of the indexed corpus, not to be accurate.
- Use full technical terms once (e.g. "James Webb Space Telescope" rather \
than just "JWST" the first time).
- Output ONLY the paragraph. Do not preface with "Answer:" or quote the question.

Question: {question}
"""


class HyDERewriter(BaseQueryRewriter):
    """LLM-generated hypothetical answer paragraph as a second query."""

    name = "hyde"

    def __init__(self, llm: LLMClient, *, temperature: float = 0.3) -> None:
        self.llm = llm
        self.temperature = temperature

    async def rewrite(self, query: str) -> list[str]:
        """Return [original_query, hypothetical_answer]. Falls back to
        [original_query] if generation fails."""
        try:
            hypothetical = await self.llm.complete(
                [{"role": "user", "content": _PROMPT.format(question=query)}],
                temperature=self.temperature,
            )
        except Exception as exc:  # noqa: BLE001
            print(f"    [hyde-fail] {exc}; using original only")
            return [query]
        hypothetical = (hypothetical or "").strip()
        if not hypothetical:
            return [query]
        return [query, hypothetical]

## §9 Rerankers

Cross-encoder reranker via the LiteLLM proxy `/v1/rerank` endpoint (Cohere-compatible).

In [ ]:
"""Reranker interface: given (query, candidates), return reordered candidates."""

from __future__ import annotations

from abc import ABC, abstractmethod



class BaseReranker(ABC):
    name: str = "base"

    @abstractmethod
    async def rerank(
        self,
        query: str,
        candidates: list[RetrievedChunk],
        *,
        top_n: int,
    ) -> list[RetrievedChunk]:
        """Return up to `top_n` candidates, sorted by reranker score desc.

        The returned RetrievedChunk.score is the reranker score; the original
        cosine is stashed in metadata['vector_score'] for inspection.
        Ranks are reset 1..top_n.
        """

In [ ]:
"""Cross-encoder reranker via the LiteLLM proxy.

Config (in litellm.yaml):

    - model_name: astrolearn-reranker
      litellm_params:
        model: jina_ai/jina-reranker-v2-base-multilingual
        api_key: os.environ/JINA_API_KEY

So this class is provider-agnostic at the call site — swap to cohere /
huggingface by changing the proxy alias only.
"""

from __future__ import annotations

from dataclasses import replace



class LiteLLMReranker(BaseReranker):
    name = "litellm_reranker"

    def __init__(self, llm: LLMClient) -> None:
        self.llm = llm

    async def rerank(
        self,
        query: str,
        candidates: list[RetrievedChunk],
        *,
        top_n: int,
    ) -> list[RetrievedChunk]:
        if not candidates:
            return []
        docs = [c.text for c in candidates]
        ranked = await self.llm.rerank(query, docs, top_n=top_n)

        out: list[RetrievedChunk] = []
        for new_rank, (orig_idx, score) in enumerate(ranked, start=1):
            src = candidates[orig_idx]
            merged_meta = {
                **src.metadata,
                "vector_score": src.score,
                "rerank_score": score,
            }
            out.append(
                replace(src, score=score, rank=new_rank, metadata=merged_meta)
            )
        return out

## §10 Reciprocal Rank Fusion (`retrieval/fusers/rrf.py`)

Used by MultiQuery and HyDE pipelines to merge several ranked lists into one.

In [ ]:
"""Reciprocal Rank Fusion — merge several ranked lists into one.

Formula: score(d) = sum over rankings R: 1 / (k + rank_R(d))
where rank is 1-indexed; documents missing from a ranking contribute 0.
k=60 is the standard from Cormack et al. 2009.

We key documents by chunk_id (assumes all rankings share the same id space,
which is true within one pipeline / one collection).
"""

from __future__ import annotations

from collections import defaultdict
from dataclasses import replace



def reciprocal_rank_fusion(
    rankings: list[list[RetrievedChunk]],
    *,
    k: int = 60,
) -> list[RetrievedChunk]:
    """Merge `rankings` via RRF; returns a single ranked list (rank reset 1-N).

    The returned RetrievedChunk.score is the RRF score (not cosine).
    The original cosine is stashed in `metadata['orig_score']` for inspection.
    """
    if not rankings:
        return []

    scores: dict[str, float] = defaultdict(float)
    representative: dict[str, RetrievedChunk] = {}

    for ranking in rankings:
        for r in ranking:
            cid = r.chunk_id or f"{r.doc_id}:{r.rank}"  # defensive fallback
            scores[cid] += 1.0 / (k + r.rank)
            representative.setdefault(cid, r)

    ordered = sorted(scores.items(), key=lambda item: -item[1])
    out: list[RetrievedChunk] = []
    for new_rank, (cid, score) in enumerate(ordered, start=1):
        src = representative[cid]
        merged_meta = {**src.metadata, "orig_score": src.score, "rrf_score": score}
        out.append(
            replace(src, score=score, rank=new_rank, metadata=merged_meta)
        )
    return out

## §11 Pipelines

A pipeline owns its Qdrant collection plus a composition of the components above. Loaded in dependency order: `base` → `char_window` (baseline) → `recursive` → `multi_query` → `recursive_reranker` → `contextual` → `hyde`. The `registry` resolves names from the CLI.

In [ ]:
"""Base RAG pipeline — owns its own qdrant collection + retrieval logic.

A Pipeline = (chunker + retriever + optional rewriter + optional reranker).
We don't model the optional pieces in the base class — each concrete
pipeline composes them however it wants. The base just standardises:

  - `name` (becomes the qdrant collection suffix)
  - `index(docs)` writes vectors
  - `retrieve(query, top_k)` returns a list of RetrievedChunks
  - `clear()` drops the collection

This keeps the contract minimal and lets pipelines diverge wildly inside.
"""

from __future__ import annotations

import uuid
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Any



@dataclass
class RetrievedChunk:
    """What a pipeline returns from .retrieve(). Includes provenance for
    the matcher to check (doc_id, page, text)."""

    doc_id: uuid.UUID
    text: str
    score: float                  # [0, 1]
    rank: int                     # 1-indexed
    page: int | None = None
    chunk_id: str | None = None
    metadata: dict[str, Any] = field(default_factory=dict)


class BasePipeline(ABC):
    name: str = ""                          # MUST override; used as collection suffix
    collection_prefix: str = "rag_retrieval_eval"

    def __init__(self, *, llm: LLMClient | None = None) -> None:
        self.llm = llm or get_llm_client()
        self._store: QdrantStore | None = None

    @property
    def store(self) -> QdrantStore:
        if self._store is None:
            self._store = QdrantStore(self.collection_name)
        return self._store

    @property
    def collection_name(self) -> str:
        if not self.name:
            raise NotImplementedError("Pipeline must set .name")
        return f"{self.collection_prefix}_{self.name}"

    @abstractmethod
    async def index(self, docs: list[CorpusDoc]) -> int:
        """Chunk + embed + upsert all docs. Returns total chunks indexed."""

    @abstractmethod
    async def retrieve(self, query: str, *, top_k: int) -> list[RetrievedChunk]:
        """Top-k retrieval; pipeline-specific composition (rewrite, rerank, etc)."""

    async def clear(self) -> None:
        """Drop this pipeline's collection (idempotent)."""
        await self.store.drop()

    def config_snapshot(self) -> dict[str, Any]:
        """Override to expose tunables in eval result JSON."""
        return {"name": self.name, "collection": self.collection_name}

In [ ]:
"""Baseline pipeline: char-window chunking + plain vector retrieval.

Mirrors the current AstroLearn backend behaviour. Numbers we get here
should match what backend produces on the same corpus + dataset, modulo
deterministic doc_ids.
"""

from __future__ import annotations

from typing import Any



class CharWindowPipeline(BasePipeline):
    name = "baseline"

    def __init__(self, *, chunk_size: int = 1000, overlap: int = 200) -> None:
        super().__init__()
        self.chunker = CharWindowChunker(size=chunk_size, overlap=overlap)
        self.chunk_size = chunk_size
        self.overlap = overlap

    def config_snapshot(self) -> dict[str, Any]:
        return {
            **super().config_snapshot(),
            "chunker": self.chunker.name,
            "chunk_size": self.chunk_size,
            "overlap": self.overlap,
        }

    async def index(self, docs: list[CorpusDoc]) -> int:
        # Fresh start each index — cheap and avoids stale entries.
        await self.store.drop()

        total = 0
        for doc in docs:
            chunks = self.chunker.chunk(doc)
            if not chunks:
                continue
            embeddings = await self.llm.embed([c.text for c in chunks])
            await self.store.ensure_collection(dimension=len(embeddings[0]))
            stored = [
                StoredChunk(
                    chunk_id=c.chunk_id,
                    doc_id=c.doc_id,
                    text=c.text,
                    vector=vec,
                    payload={
                        "page": c.page,
                        "char_offset": c.char_offset,
                        **c.metadata,
                    },
                )
                for c, vec in zip(chunks, embeddings, strict=True)
            ]
            total += await self.store.upsert(stored)
            print(f"  [{doc.filename}] {len(chunks)} chunks")
        return total

    async def retrieve(self, query: str, *, top_k: int) -> list[RetrievedChunk]:
        embedding = (await self.llm.embed([query]))[0]
        hits = await self.store.query(embedding, top_k=top_k)
        return [
            RetrievedChunk(
                doc_id=h.doc_id,
                text=h.text,
                score=h.score,
                rank=i + 1,
                page=_coerce_page(h.payload.get("page")),
                chunk_id=h.chunk_id,
                metadata=h.payload,
            )
            for i, h in enumerate(hits)
        ]


def _coerce_page(raw: Any) -> int | None:
    if raw is None:
        return None
    try:
        return int(raw)
    except (TypeError, ValueError):
        return None


@register("baseline")
def _factory() -> CharWindowPipeline:
    return CharWindowPipeline(chunk_size=1000, overlap=200)

In [ ]:
"""Pipeline = recursive splitter + plain vector retrieval.

Only difference from `baseline` is the chunker. Same retrieve(), same store.
This isolates the chunking-strategy variable for clean ablation.
"""

from __future__ import annotations

from typing import Any



class RecursivePipeline(BasePipeline):
    name = "recursive"

    def __init__(self, *, chunk_size: int = 1000, overlap: int = 150) -> None:
        super().__init__()
        self.chunker = RecursiveChunker(size=chunk_size, overlap=overlap)
        self.chunk_size = chunk_size
        self.overlap = overlap

    def config_snapshot(self) -> dict[str, Any]:
        return {
            **super().config_snapshot(),
            "chunker": self.chunker.name,
            "chunk_size": self.chunk_size,
            "overlap": self.overlap,
            "separators": list(self.chunker.separators),
        }

    async def index(self, docs: list[CorpusDoc]) -> int:
        await self.store.drop()

        total = 0
        for doc in docs:
            chunks = self.chunker.chunk(doc)
            if not chunks:
                continue
            embeddings = await self.llm.embed([c.text for c in chunks])
            await self.store.ensure_collection(dimension=len(embeddings[0]))
            stored = [
                StoredChunk(
                    chunk_id=c.chunk_id,
                    doc_id=c.doc_id,
                    text=c.text,
                    vector=vec,
                    payload={
                        "page": c.page,
                        "char_offset": c.char_offset,
                        **c.metadata,
                    },
                )
                for c, vec in zip(chunks, embeddings, strict=True)
            ]
            total += await self.store.upsert(stored)
            print(f"  [{doc.filename}] {len(chunks)} chunks")
        return total

    async def retrieve(self, query: str, *, top_k: int) -> list[RetrievedChunk]:
        embedding = (await self.llm.embed([query]))[0]
        hits = await self.store.query(embedding, top_k=top_k)
        return [
            RetrievedChunk(
                doc_id=h.doc_id,
                text=h.text,
                score=h.score,
                rank=i + 1,
                page=_coerce_page(h.payload.get("page")),
                chunk_id=h.chunk_id,
                metadata=h.payload,
            )
            for i, h in enumerate(hits)
        ]


def _coerce_page(raw: Any) -> int | None:
    if raw is None:
        return None
    try:
        return int(raw)
    except (TypeError, ValueError):
        return None


@register("recursive")
def _factory() -> RecursivePipeline:
    return RecursivePipeline(chunk_size=1000, overlap=150)


@register("tuned_recursive")
def _tuned_factory() -> RecursivePipeline:
    """Recursive with hyperparams chosen by sweep_chunker.py against
    arxiv_curated_v1: smaller chunks dominate for factual/numeric queries.
    Best MRR + hit@1 + hit@10 in the 12-config grid."""
    return RecursivePipeline(chunk_size=500, overlap=250)

In [ ]:
"""Recursive chunker + multi-query rewrite + RRF fusion.

Isolates the "query analysis" variable: indexing is identical to `recursive`,
only retrieve() changes. Same Qdrant collection NOT shared (each pipeline owns
its own), but the chunking strategy is the same — so a head-to-head with
`recursive` cleanly measures the rewriter's contribution.
"""

from __future__ import annotations

import asyncio
from typing import Any



class MultiQueryRecursivePipeline(RecursivePipeline):
    name = "recursive_multiquery"

    def __init__(
        self,
        *,
        chunk_size: int = 1000,
        overlap: int = 150,
        n_variants: int = 3,
        sub_top_k_multiplier: int = 2,
        rrf_k: int = 60,
        rewriter_temperature: float = 0.5,
    ) -> None:
        super().__init__(chunk_size=chunk_size, overlap=overlap)
        self.rewriter = MultiQueryRewriter(
            self.llm, n_variants=n_variants, temperature=rewriter_temperature
        )
        self.n_variants = n_variants
        self.sub_top_k_multiplier = sub_top_k_multiplier
        self.rrf_k = rrf_k

    def config_snapshot(self) -> dict[str, Any]:
        return {
            **super().config_snapshot(),
            "rewriter": self.rewriter.name,
            "n_variants": self.n_variants,
            "sub_top_k_multiplier": self.sub_top_k_multiplier,
            "rrf_k": self.rrf_k,
        }

    async def retrieve(self, query: str, *, top_k: int) -> list[RetrievedChunk]:
        variants = await self.rewriter.rewrite(query)
        # Pull a wider net per variant so RRF has redundancy to work with.
        sub_k = top_k * self.sub_top_k_multiplier

        # Parallelise sub-retrievals; each is one embed + one qdrant call.
        rankings = await asyncio.gather(
            *(self._retrieve_single(v, sub_k) for v in variants)
        )
        fused = reciprocal_rank_fusion(rankings, k=self.rrf_k)
        return fused[:top_k]

    async def _retrieve_single(
        self, query: str, top_k: int
    ) -> list[RetrievedChunk]:
        embedding = (await self.llm.embed([query]))[0]
        hits = await self.store.query(embedding, top_k=top_k)
        return [
            RetrievedChunk(
                doc_id=h.doc_id,
                text=h.text,
                score=h.score,
                rank=i + 1,
                page=_coerce_page(h.payload.get("page")),
                chunk_id=h.chunk_id,
                metadata=h.payload,
            )
            for i, h in enumerate(hits)
        ]


@register("recursive_multiquery")
def _factory() -> MultiQueryRecursivePipeline:
    return MultiQueryRecursivePipeline(
        chunk_size=1000,
        overlap=150,
        n_variants=3,
        sub_top_k_multiplier=2,
        rrf_k=60,
    )

In [ ]:
"""Recursive chunker + vector search + cross-encoder reranker.

Two-stage retrieval:
  1. Vector search pulls a wider pool (top_k * candidate_multiplier)
  2. Reranker re-orders that pool and returns top_k

Same chunker as `recursive` — isolates the reranker variable cleanly.
"""

from __future__ import annotations

from typing import Any



class RecursiveRerankerPipeline(RecursivePipeline):
    name = "recursive_reranker"

    def __init__(
        self,
        *,
        chunk_size: int = 1000,
        overlap: int = 150,
        candidate_multiplier: int = 4,
    ) -> None:
        super().__init__(chunk_size=chunk_size, overlap=overlap)
        self.reranker = LiteLLMReranker(self.llm)
        self.candidate_multiplier = candidate_multiplier

    def config_snapshot(self) -> dict[str, Any]:
        return {
            **super().config_snapshot(),
            "reranker": self.reranker.name,
            "candidate_multiplier": self.candidate_multiplier,
        }

    async def retrieve(self, query: str, *, top_k: int) -> list[RetrievedChunk]:
        # Stage 1: wide vector pool.
        candidates = await super().retrieve(
            query, top_k=top_k * self.candidate_multiplier
        )
        # Stage 2: rerank with cross-encoder.
        return await self.reranker.rerank(query, candidates, top_n=top_k)


@register("recursive_reranker")
def _factory() -> RecursiveRerankerPipeline:
    return RecursiveRerankerPipeline(
        chunk_size=1000, overlap=150, candidate_multiplier=4
    )


@register("tuned_recursive_reranker")
def _tuned_factory() -> RecursiveRerankerPipeline:
    """Tuned chunker hyperparams (from sweep_chunker.py) + reranker.
    This is the recommended production config."""
    return RecursiveRerankerPipeline(
        chunk_size=500, overlap=250, candidate_multiplier=4
    )

In [ ]:
"""Recursive + contextual chunker + plain vector retrieval.

Isolates the contextual-chunking variable. Compared to `recursive`:
  - Same recursive splitter
  - Each chunk text is prefixed with the paper's filename + first line of text
  - The prefix is included when embedding (improves cross-doc disambiguation)
  - The prefix is NOT included in citations / context shown to LLM (we keep
    the original chunk.text for that — the prefix is purely a retrieval-time
    augmentation).
"""

from __future__ import annotations

from typing import Any



class ContextualRecursivePipeline(BasePipeline):
    name = "contextual_recursive"

    def __init__(self, *, chunk_size: int = 500, overlap: int = 250) -> None:
        super().__init__()
        self.chunker = ContextualChunker(
            inner=RecursiveChunker(size=chunk_size, overlap=overlap)
        )
        self.chunk_size = chunk_size
        self.overlap = overlap

    def config_snapshot(self) -> dict[str, Any]:
        return {
            **super().config_snapshot(),
            "chunker": self.chunker.name,
            "chunk_size": self.chunk_size,
            "overlap": self.overlap,
        }

    async def index(self, docs: list[CorpusDoc]) -> int:
        await self.store.drop()

        total = 0
        for doc in docs:
            chunks = self.chunker.chunk(doc)
            if not chunks:
                continue
            # Embed the prefixed text; store the prefixed text in payload too
            # so callers see what was indexed. The page + char_offset map back
            # to the original doc.full_text for page-citation purposes.
            embeddings = await self.llm.embed([c.text for c in chunks])
            await self.store.ensure_collection(dimension=len(embeddings[0]))
            stored = [
                StoredChunk(
                    chunk_id=c.chunk_id,
                    doc_id=c.doc_id,
                    text=c.text,
                    vector=vec,
                    payload={
                        "page": c.page,
                        "char_offset": c.char_offset,
                        **c.metadata,
                    },
                )
                for c, vec in zip(chunks, embeddings, strict=True)
            ]
            total += await self.store.upsert(stored)
            print(f"  [{doc.filename}] {len(chunks)} chunks")
        return total

    async def retrieve(self, query: str, *, top_k: int) -> list[RetrievedChunk]:
        embedding = (await self.llm.embed([query]))[0]
        hits = await self.store.query(embedding, top_k=top_k)
        return [
            RetrievedChunk(
                doc_id=h.doc_id,
                text=h.text,
                score=h.score,
                rank=i + 1,
                page=_coerce_page(h.payload.get("page")),
                chunk_id=h.chunk_id,
                metadata=h.payload,
            )
            for i, h in enumerate(hits)
        ]


class ContextualRecursiveRerankerPipeline(ContextualRecursivePipeline):
    """Step 4: contextual + recursive + cross-encoder reranker."""

    name = "contextual_recursive_reranker"

    def __init__(
        self,
        *,
        chunk_size: int = 1500,
        overlap: int = 250,
        candidate_multiplier: int = 4,
    ) -> None:
        super().__init__(chunk_size=chunk_size, overlap=overlap)
        # Local import to avoid eager-loading rerankers in non-rerank runs.
        self.reranker = LiteLLMReranker(self.llm)
        self.candidate_multiplier = candidate_multiplier

    def config_snapshot(self) -> dict[str, Any]:
        return {
            **super().config_snapshot(),
            "reranker": self.reranker.name,
            "candidate_multiplier": self.candidate_multiplier,
        }

    async def retrieve(self, query: str, *, top_k: int) -> list[RetrievedChunk]:
        candidates = await super().retrieve(
            query, top_k=top_k * self.candidate_multiplier
        )
        return await self.reranker.rerank(query, candidates, top_n=top_k)


@register("contextual_recursive")
def _ctx_factory() -> ContextualRecursivePipeline:
    return ContextualRecursivePipeline(chunk_size=500, overlap=250)


@register("contextual_recursive_reranker")
def _ctx_rerank_factory() -> ContextualRecursiveRerankerPipeline:
    return ContextualRecursiveRerankerPipeline(
        chunk_size=500, overlap=250, candidate_multiplier=4
    )

In [ ]:
"""Tuned recursive + HyDE + reranker pipeline (Step 3 of the monotonic ladder).

Flow:
  1. HyDE rewrites the user question into [original, hypothetical_answer]
  2. Embed + qdrant-search each separately
  3. RRF-merge into a single candidate list (de-duped by chunk_id)
  4. Cross-encoder reranker re-orders against the ORIGINAL query
  5. Return top_k

Why the ORIGINAL query goes to the reranker (not the hypothetical):
  HyDE helps RECALL (broader candidate set). The reranker handles PRECISION
  using the actual user intent. Reranking against a fabricated answer would
  bias toward hallucinations.

Shares chunker config with tuned_recursive_reranker so this isolates the
HyDE contribution cleanly.
"""

from __future__ import annotations

import asyncio
from typing import Any



class TunedRecursiveHyDERerankerPipeline(RecursiveRerankerPipeline):
    name = "tuned_recursive_hyde_reranker"

    def __init__(
        self,
        *,
        chunk_size: int = 500,
        overlap: int = 250,
        candidate_multiplier: int = 4,
        rrf_k: int = 60,
        hyde_temperature: float = 0.3,
    ) -> None:
        super().__init__(
            chunk_size=chunk_size,
            overlap=overlap,
            candidate_multiplier=candidate_multiplier,
        )
        self.rewriter = HyDERewriter(self.llm, temperature=hyde_temperature)
        self.rrf_k = rrf_k

    def config_snapshot(self) -> dict[str, Any]:
        return {
            **super().config_snapshot(),
            "rewriter": self.rewriter.name,
            "rrf_k": self.rrf_k,
            "hyde_temperature": self.rewriter.temperature,
        }

    async def retrieve(self, query: str, *, top_k: int) -> list[RetrievedChunk]:
        # Stage 1: HyDE expansion → 1-2 queries.
        variants = await self.rewriter.rewrite(query)
        sub_k = top_k * self.candidate_multiplier

        # Stage 2: parallel vector search per variant.
        rankings = await asyncio.gather(
            *(self._retrieve_single(v, sub_k) for v in variants)
        )
        # Stage 3: RRF merge (de-duplicates by chunk_id).
        merged = reciprocal_rank_fusion(rankings, k=self.rrf_k)
        # Cap the candidates fed into the reranker so we don't pay rerank
        # cost on the long tail; same effective candidate budget as the
        # plain reranker pipeline.
        candidates = merged[: sub_k]

        # Stage 4: rerank against ORIGINAL query, not the HyDE expansion.
        return await self.reranker.rerank(query, candidates, top_n=top_k)

    async def _retrieve_single(
        self, query: str, top_k: int
    ) -> list[RetrievedChunk]:
        embedding = (await self.llm.embed([query]))[0]
        hits = await self.store.query(embedding, top_k=top_k)
        return [
            RetrievedChunk(
                doc_id=h.doc_id,
                text=h.text,
                score=h.score,
                rank=i + 1,
                page=_coerce_page(h.payload.get("page")),
                chunk_id=h.chunk_id,
                metadata=h.payload,
            )
            for i, h in enumerate(hits)
        ]


@register("tuned_recursive_hyde_reranker")
def _factory() -> TunedRecursiveHyDERerankerPipeline:
    return TunedRecursiveHyDERerankerPipeline(
        chunk_size=500,
        overlap=250,
        candidate_multiplier=4,
        rrf_k=60,
        hyde_temperature=0.3,
    )

In [ ]:
"""Registry of named pipelines for CLI dispatch.

CLI says `run index baseline` -> registry lookup -> instantiate.
Keeping this central means run.py doesn't need to know about each pipeline,
and adding a new pipeline = one import + one register() call.
"""

from __future__ import annotations

from typing import Callable


PipelineFactory = Callable[[], BasePipeline]

_REGISTRY: dict[str, PipelineFactory] = {}


def register(name: str) -> Callable[[PipelineFactory], PipelineFactory]:
    """Decorator: register a pipeline factory under `name`."""
    def _wrap(factory: PipelineFactory) -> PipelineFactory:
        if name in _REGISTRY:
            raise ValueError(f"Pipeline already registered: {name}")
        _REGISTRY[name] = factory
        return factory
    return _wrap


def get(name: str) -> BasePipeline:
    if name not in _REGISTRY:
        raise KeyError(
            f"Unknown pipeline '{name}'. Available: {sorted(_REGISTRY)}"
        )
    return _REGISTRY[name]()


def names() -> list[str]:
    return sorted(_REGISTRY)


# Side-effect imports register the pipelines.
# Add new pipelines below as they're implemented.

## §12 Eval

`matcher.is_relevant()` checks `(doc_id, page ± tolerance, contains_substring)`. Metrics are `hit_rate@k`, `recall@k`, `MRR` over per-query hit ranks.

In [ ]:
"""Relevance check: does a retrieved chunk satisfy a golden constraint?"""

from __future__ import annotations



def _normalize(text: str) -> str:
    """Lowercase + collapse whitespace — survives pypdf newline glitches."""
    return " ".join(text.split()).lower()


def is_relevant(match: RetrievedChunk, golden: GoldenChunk) -> bool:
    if match.doc_id != golden.doc_id:
        return False
    if golden.page is not None:
        if match.page is None:
            return False
        if abs(match.page - golden.page) > golden.page_tolerance:
            return False
    if golden.contains:
        needle = _normalize(golden.contains)
        if needle and needle not in _normalize(match.text):
            return False
    return True


def hit_ranks(
    matches: list[RetrievedChunk], relevants: list[GoldenChunk]
) -> list[int]:
    """1-indexed ranks of `matches` that satisfy any of `relevants`."""
    ranks: list[int] = []
    for m in matches:
        if any(is_relevant(m, g) for g in relevants):
            ranks.append(m.rank)
    return ranks

In [ ]:
"""Pure metric functions over per-query hit ranks."""

from __future__ import annotations

from collections.abc import Sequence


def hit_rate_at_k(hit_ranks_list: Sequence[Sequence[int]], k: int) -> float:
    """Fraction of queries with at least one relevant chunk in top-k."""
    if not hit_ranks_list:
        return 0.0
    hits = sum(1 for r in hit_ranks_list if any(x <= k for x in r))
    return hits / len(hit_ranks_list)


def recall_at_k(
    hit_ranks_list: Sequence[Sequence[int]],
    n_relevant_list: Sequence[int],
    k: int,
) -> float:
    """Avg per-query fraction of relevants found in top-k."""
    if not hit_ranks_list or len(hit_ranks_list) != len(n_relevant_list):
        return 0.0
    total = 0.0
    counted = 0
    for ranks, n_rel in zip(hit_ranks_list, n_relevant_list, strict=True):
        if n_rel <= 0:
            continue
        hits = sum(1 for r in ranks if r <= k)
        total += min(hits, n_rel) / n_rel
        counted += 1
    return total / counted if counted > 0 else 0.0


def mrr(hit_ranks_list: Sequence[Sequence[int]]) -> float:
    """Mean Reciprocal Rank: avg of 1/first_rank (0 for total misses)."""
    if not hit_ranks_list:
        return 0.0
    total = 0.0
    for r in hit_ranks_list:
        if r:
            total += 1.0 / min(r)
    return total / len(hit_ranks_list)

In [ ]:
"""Run a pipeline over a GoldenDataset, compute metrics."""

from __future__ import annotations

from datetime import UTC, datetime
from pathlib import Path

import yaml

    EvalResult,
    GoldenDataset,
    GoldenQuery,
    QueryResult,
)


def load_dataset(path: Path) -> GoldenDataset:
    with path.open("r", encoding="utf-8") as f:
        raw = yaml.safe_load(f)
    return GoldenDataset.model_validate(raw)


async def evaluate_query(
    query: GoldenQuery, pipeline: BasePipeline, *, top_k: int
) -> QueryResult:
    matches = await pipeline.retrieve(query.question, top_k=top_k)
    ranks = hit_ranks(matches, query.relevant)
    return QueryResult(
        query_id=query.id,
        question=query.question,
        n_relevant=len(query.relevant),
        n_retrieved=len(matches),
        hit_ranks=ranks,
        first_hit_rank=min(ranks) if ranks else None,
    )


async def run_eval(
    dataset: GoldenDataset,
    pipeline: BasePipeline,
    *,
    dataset_path: str,
    top_k: int = 10,
) -> EvalResult:
    results: list[QueryResult] = []
    for query in dataset.queries:
        qr = await evaluate_query(query, pipeline, top_k=top_k)
        results.append(qr)

    hr = [r.hit_ranks for r in results]
    n_rel = [r.n_relevant for r in results]

    return EvalResult(
        pipeline_name=pipeline.name,
        dataset_name=dataset.name,
        dataset_path=dataset_path,
        timestamp=datetime.now(UTC),
        top_k=top_k,
        pipeline_config=pipeline.config_snapshot(),
        queries=results,
        hit_rate_at_1=hit_rate_at_k(hr, 1),
        hit_rate_at_5=hit_rate_at_k(hr, 5),
        hit_rate_at_10=hit_rate_at_k(hr, 10),
        recall_at_5=recall_at_k(hr, n_rel, 5),
        recall_at_10=recall_at_k(hr, n_rel, 10),
        mrr=mrr(hr),
    )


def print_summary(r: EvalResult) -> None:
    print()
    print(f"=== {r.pipeline_name} on {r.dataset_name} ===")
    print(f"  queries:       {r.n_queries}")
    print(f"  top_k:         {r.top_k}")
    print(f"  hit_rate@1:    {r.hit_rate_at_1:.3f}")
    print(f"  hit_rate@5:    {r.hit_rate_at_5:.3f}")
    print(f"  hit_rate@10:   {r.hit_rate_at_10:.3f}")
    print(f"  recall@5:      {r.recall_at_5:.3f}")
    print(f"  recall@10:     {r.recall_at_10:.3f}")
    print(f"  MRR:           {r.mrr:.3f}")
    misses = [q.query_id for q in r.queries if q.first_hit_rank is None]
    if misses:
        head = ", ".join(misses[:8])
        tail = " ..." if len(misses) > 8 else ""
        print(f"  total misses ({len(misses)}): {head}{tail}")

In [ ]:
"""Side-by-side comparison of two EvalResults."""

from __future__ import annotations

import json
from pathlib import Path



def load_result(path: Path) -> EvalResult:
    return EvalResult.model_validate(json.loads(path.read_text(encoding="utf-8")))


def compare(baseline: EvalResult, candidate: EvalResult) -> None:
    if baseline.dataset_name != candidate.dataset_name:
        print(
            f"[!] datasets differ: {baseline.dataset_name} vs "
            f"{candidate.dataset_name} — comparison is meaningless."
        )
        return

    print()
    print(f"=== {baseline.pipeline_name}  ->  {candidate.pipeline_name} ===")
    print(f"  {'metric':<16} {'baseline':>10} {'candidate':>10} {'delta':>10}")
    for k in metric_keys():
        bv, cv = getattr(baseline, k), getattr(candidate, k)
        d = cv - bv
        sign = "+" if d > 0 else ""
        print(f"  {k:<16} {bv:>10.3f} {cv:>10.3f} {sign}{d:>9.3f}")

    b_by_id = {q.query_id: q for q in baseline.queries}
    c_by_id = {q.query_id: q for q in candidate.queries}
    common = sorted(set(b_by_id) & set(c_by_id))
    regressions = [
        q for q in common
        if b_by_id[q].first_hit_rank is not None
        and c_by_id[q].first_hit_rank is None
    ]
    new_hits = [
        q for q in common
        if b_by_id[q].first_hit_rank is None
        and c_by_id[q].first_hit_rank is not None
    ]
    if regressions:
        print(f"\n  regressions ({len(regressions)}): {', '.join(regressions)}")
    if new_hits:
        print(f"  new hits    ({len(new_hits)}): {', '.join(new_hits)}")

## §13 Chunker hyperparam sweep (`sweep_chunker.py`)

Grid over `(size ∈ {500, 750, 1000, 1500}) × (overlap ∈ {100, 200, 250})`. Output of this sweep picked the production hyperparams: `RecursiveChunker(size=500, overlap=250)`.

In [ ]:
"""Hyperparameter sweep for recursive chunker — find config that beats baseline.

Iterates over (size, overlap) grid. For each:
  1. Build a one-off pipeline with that config (in-memory, not registered)
  2. Index into a unique throwaway collection
  3. Run eval against the curated dataset
  4. Drop collection
  5. Print row of metrics

Usage:
    python -m sweep_chunker --dataset dataset/datasets/arxiv_curated_v1.yaml
"""

from __future__ import annotations

import argparse
import asyncio
import sys
from pathlib import Path

from dotenv import load_dotenv
load_dotenv(Path(__file__).resolve().parent / ".env")

# Import registry first to break the circular load order (registry imports
# all pipeline modules; afterwards we can import any of them directly).


SIZES = [500, 750, 1000, 1500]
OVERLAPS = [100, 200, 250]


class _SweepPipeline(RecursivePipeline):
    """Recursive pipeline with a unique collection per (size, overlap)."""

    def __init__(self, *, size: int, overlap: int) -> None:
        super().__init__(chunk_size=size, overlap=overlap)
        self._suffix = f"sweep_r{size}_{overlap}"

    @property
    def collection_name(self) -> str:
        return f"{self.collection_prefix}_{self._suffix}"


async def _eval_one(size: int, overlap: int, dataset_path: Path) -> dict:
    pipeline = _SweepPipeline(size=size, overlap=overlap)
    docs = parse_corpus()
    n = await pipeline.index(docs)
    dataset = load_dataset(dataset_path)
    result = await run_eval(dataset, pipeline, dataset_path=str(dataset_path), top_k=10)
    await pipeline.clear()
    return {
        "size": size,
        "overlap": overlap,
        "chunks": n,
        "hit_1": result.hit_rate_at_1,
        "hit_5": result.hit_rate_at_5,
        "hit_10": result.hit_rate_at_10,
        "recall_5": result.recall_at_5,
        "mrr": result.mrr,
    }


async def main() -> int:
    p = argparse.ArgumentParser()
    p.add_argument(
        "--dataset", default="dataset/datasets/arxiv_curated_v1.yaml",
    )
    args = p.parse_args()

    dataset_path = Path(args.dataset).resolve()
    rows: list[dict] = []
    for size in SIZES:
        for overlap in OVERLAPS:
            if overlap >= size:
                continue
            print(f"[*] running recursive size={size} overlap={overlap}", flush=True)
            row = await _eval_one(size, overlap, dataset_path)
            rows.append(row)
            print(
                f"  hit@1={row['hit_1']:.3f}  hit@5={row['hit_5']:.3f}  "
                f"hit@10={row['hit_10']:.3f}  MRR={row['mrr']:.3f}  "
                f"chunks={row['chunks']}",
                flush=True,
            )

    rows.sort(key=lambda r: (-r["hit_5"], -r["mrr"]))
    print("\n=== SWEEP RESULTS (sorted by hit@5 desc, then MRR) ===")
    print(f"  {'size':>5}  {'overlap':>7}  {'chunks':>6}  {'hit@1':>6}  {'hit@5':>6}  {'hit@10':>6}  {'MRR':>6}")
    for r in rows:
        print(
            f"  {r['size']:>5}  {r['overlap']:>7}  {r['chunks']:>6}  "
            f"{r['hit_1']:>6.3f}  {r['hit_5']:>6.3f}  {r['hit_10']:>6.3f}  {r['mrr']:>6.3f}"
        )
    return 0


if __name__ == "__main__":
    sys.exit(asyncio.run(main()))

## §14 Run the pipeline (optional)

```python
import asyncio
from pathlib import Path

# 1. Download corpus PDFs (~13 MB total)
download_all()

# 2. Parse PDFs → in-memory CorpusDoc + corpus/manifest.json
docs = parse_corpus()

# 3. Generate golden dataset (per-page LLM prompt)
asyncio.run(generate(per_doc=6, name='arxiv_v1'))

# 4. Index + eval one pipeline
async def go(name: str):
    pipeline = get(name)
    docs = parse_corpus()
    await pipeline.index(docs)
    dataset = load_dataset(Path('rag_retrieval_eval_results/datasets/arxiv_curated_v1.yaml'))
    result = await run_eval(dataset, pipeline, dataset_path='arxiv_curated_v1.yaml', top_k=10)
    print_summary(result)

# Compare baseline → tuned chunker → +reranker
for pl in ('baseline', 'tuned_recursive', 'tuned_recursive_reranker'):
    asyncio.run(go(pl))
```


## §15 Final result — pipeline comparison table

The cached comparison lives in `rag_retrieval_eval_results/comparison.md`. One table, the 3-step monotonic ladder (baseline → tuned chunker → + reranker) — the headline numbers for the report.

In [ ]:
from IPython.display import Markdown, display
from pathlib import Path
display(Markdown((Path('rag_retrieval_eval_results') / 'comparison.md').read_text(encoding='utf-8')))

## §16 Dataset reference

The ladder was evaluated on `arxiv_curated_v1` — 30 hand-curated grounded questions over 6 arXiv astronomy papers. Each question pinned by `(doc_id, page ±1, contains_substring)` so the dataset survives chunker reshapes.

In [ ]:
import yaml
ds = yaml.safe_load((Path('rag_retrieval_eval_results') / 'datasets' / 'arxiv_curated_v1.yaml').read_text(encoding='utf-8'))
print(f"Dataset: {ds['name']} — {len(ds['queries'])} queries\n")
for q in ds['queries'][:5]:
    rel = q['relevant'][0]
    print(f"  [{q['id']}] {q['question']}")
    print(f"        → doc={rel['doc_id'][:8]}…  page={rel['page']}  needle={rel['contains'][:60]!r}")